In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import pyvisa
import time
import os
from datetime import datetime
import nysg_tools as ny

In [ ]:
# --- Frequency sweep segments (Hz) ---
segments = [
    (49500, 50050, 10.0),  # pre-resonance
    (50050, 50085,  2.0),  # near resonance
    (50085, 50105,  0.3),  # resonance (dense)
    (50105, 50265,  2.0),  # near resonance
    (50265, 50295,  0.3),  # anti-resonance (dense)
    (50295, 50350,  2.0),  # near anti-resonance
    (50350, 51000, 10.0),  # post-resonances
]

freqs = [np.arange(start, stop, step) for start, stop, step in segments]
freq_array = np.unique(np.round(np.concatenate(freqs), decimals=3))

In [34]:
rm = pyvisa.ResourceManager()
rm.list_resources()

('USB0::0x0699::0x0346::C036493::INSTR', 'ASRL1::INSTR', 'GPIB0::8::INSTR')

In [35]:
inst   = rm.open_resource("GPIB0::8::INSTR")
inst.write_termination = "\n"
inst.read_termination  = "\n"

fungen = rm.open_resource("USB0::0x0699::0x0346::C036493::INSTR")

In [69]:
def get_lockin():
    """Read X, Y, R, Theta from lock-in amplifier."""
    x, y, r, theta = map(float, inst.query("SNAP? 1,2,3,4").split(","))
    return x, y, r, theta


def setup_save_folder():
    """Create a timestamped output folder (e.g. 14h_30m_45s)."""
    folder_name = datetime.now().strftime("%Hh_%Mm_%Ss")
    os.makedirs(folder_name, exist_ok=True)
    return folder_name

In [76]:
# --- Lock-in Setup ---
inst.write("*RST")    # Reset
time.sleep(1)

inst.write("FMOD 0")  # External reference
inst.write("OFLT 8")  # Tau = 30 ms
inst.write("OFSL 3")  # Slope = 12 dB/oct
inst.write("SENS 20") # Sensitivity = 10mV
inst.write("RMOD 1")  # Normal reserve
inst.write("RSLP 1")  # Positive edge for square ref

7

In [ ]:
# De 49500 a 50060 en 10mV (20)
# De 50060 a 50090 en 50 mV (22)
# de 50090 a 50110 en 200 mV (24)
# de 50110 a 50130 en 50 mV (22)
# de 50130 a 50200 en 10mV (20)
# de 50200 a 50240 en 2mV (18)
# de 50240 a 50272 en 500uV (16)
# de 50272 a 50281 en 100uV (14)
# de 50281 a 50289 en 50uV (13)
# de 50289 a 50318 en 500uV (16)
# de 50318 a 50350 en 500uV (16)
# de 50350 a 51000 en 2mV (18)

In [77]:
SENS_RULES = [
    (49500, 50060, "SENS 20", "10 mV"),
    (50060, 50090, "SENS 22", "50 mV"),
    (50090, 50110, "SENS 24", "200 mV"),
    (50110, 50130, "SENS 22", "50 mV"),
    (50130, 50200, "SENS 20", "10 mV"),
    (50200, 50240, "SENS 18", "2 mV"),
    (50240, 50272, "SENS 16", "500 uV"),
    (50272, 50281, "SENS 14", "100 uV"),
    (50281, 50289, "SENS 13", "50 uV"),
    (50289, 50318, "SENS 16", "500 uV"),
    (50318, 50350, "SENS 16", "500 uV"),
    (50350, 51000, "SENS 18", "2 mV"),
]

SENS_OVERRIDES = {}

for start_f, _, sens_cmd, sens_val in SENS_RULES:
    # find the first index where freq >= start_f
    idx = np.searchsorted(freq_array, start_f)
    SENS_OVERRIDES[idx] = (sens_cmd, sens_val)

print(SENS_OVERRIDES)

{0: ('SENS 20', '10 mV'), 60: ('SENS 22', '50 mV'), 98: ('SENS 24', '200 mV'), 176: ('SENS 22', '50 mV'), 186: ('SENS 20', '10 mV'), 221: ('SENS 18', '2 mV'), 241: ('SENS 16', '500 uV'), 288: ('SENS 14', '100 uV'), 333: ('SENS 13', '50 uV'), 373: ('SENS 16', '500 uV'), 415: ('SENS 16', '500 uV'), 431: ('SENS 18', '2 mV')}


In [80]:
inst.close()

In [78]:
# --- Sweep Parameters ---
TAU             = 0.1   # 30 ms (matches OFLT 7)
WAIT_TIME       = 12 * TAU  # settle time after each frequency step
N_MEASUREMENTS  = 2         # repeated readings per frequency
INTER_MEAS_DT   = TAU       # short pause between repeated readings

total_points    = len(freq_array)
time_per_point  = WAIT_TIME + N_MEASUREMENTS * INTER_MEAS_DT
estimated_min   = round((total_points * time_per_point) / 60, 1)

save_dir = setup_save_folder()

print(f"🧈 Starting sweep — {total_points} points | N={N_MEASUREMENTS} meas/freq | ~{estimated_min} min")

# # Sensitivity override checkpoints (index : SENS command)
# SENS_OVERRIDES = {
#     321: ("SENS 19", "5 mV"),   # ~50200 Hz
#     493: ("SENS 17", "1 mV"),   # ~50279 Hz
#     543: ("SENS 19", "5 mV"),   # ~50284 Hz
# }

for i, freq in enumerate(freq_array):

    # Sensitivity overrides
    if i in SENS_OVERRIDES:
        cmd, label = SENS_OVERRIDES[i]
        inst.write(cmd)
        print(f"  >> Sensitivity → {label} at index {i}")

    # Set frequency
    fungen.write(f"SOURCE1:FREQ {freq}")

    # Wait for filter to settle
    time.sleep(WAIT_TIME)

    # Take N repeated measurements
    readings = []
    for _ in range(N_MEASUREMENTS):
        readings.append(get_lockin())
        time.sleep(INTER_MEAS_DT)

    readings = np.array(readings)  # shape (N, 4): X, Y, R, Theta

    means = readings.mean(axis=0)
    stds  = readings.std(axis=0, ddof=1)

    x_mean, y_mean, r_mean, theta_mean = means
    x_std,  y_std,  r_std,  theta_std  = stds

    # Save to pickle
    data_dict = {
        'index':          i,
        'frequency_Hz':   freq,
        'n_measurements': N_MEASUREMENTS,
        # Raw readings
        'X_V_raw':        readings[:, 0].tolist(),
        'Y_V_raw':        readings[:, 1].tolist(),
        'R_V_raw':        readings[:, 2].tolist(),
        'Theta_deg_raw':  readings[:, 3].tolist(),
        # Statistics
        'X_V_mean':       x_mean,    'X_V_std':       x_std,
        'Y_V_mean':       y_mean,    'Y_V_std':       y_std,
        'R_V_mean':       r_mean,    'R_V_std':       r_std,
        'Theta_deg_mean': theta_mean,'Theta_deg_std': theta_std,
    }

    # Filename: 0000_50000.000Hz.pkl
    filename = os.path.join(save_dir, f"{i:04d}_{freq:.3f}Hz.pkl")
    with open(filename, "wb") as f:
        pickle.dump(data_dict, f)

    print(f"[{i+1}/{total_points}] {filename} 🧈 | R: {r_mean:.4f} ± {r_std:.4f} V | θ: {theta_mean:.2f} ± {theta_std:.2f} deg")

🧈 Starting sweep — 496 points | N=2 meas/freq | ~11.6 min
  >> Sensitivity → 10 mV at index 0
[1/496] 11h_23m_41s\0000_49500.000Hz.pkl 🧈 | R: 0.0031 ± 0.0000 V | θ: 60.76 ± 0.00 deg
[2/496] 11h_23m_41s\0001_49510.000Hz.pkl 🧈 | R: 0.0031 ± 0.0000 V | θ: 60.76 ± 0.00 deg
[3/496] 11h_23m_41s\0002_49520.000Hz.pkl 🧈 | R: 0.0032 ± 0.0000 V | θ: 60.75 ± 0.00 deg
[4/496] 11h_23m_41s\0003_49530.000Hz.pkl 🧈 | R: 0.0032 ± 0.0000 V | θ: 60.74 ± 0.00 deg
[5/496] 11h_23m_41s\0004_49540.000Hz.pkl 🧈 | R: 0.0032 ± 0.0000 V | θ: 60.73 ± 0.00 deg
[6/496] 11h_23m_41s\0005_49550.000Hz.pkl 🧈 | R: 0.0032 ± 0.0000 V | θ: 60.73 ± 0.00 deg
[7/496] 11h_23m_41s\0006_49560.000Hz.pkl 🧈 | R: 0.0032 ± 0.0000 V | θ: 60.72 ± 0.00 deg
[8/496] 11h_23m_41s\0007_49570.000Hz.pkl 🧈 | R: 0.0032 ± 0.0000 V | θ: 60.71 ± 0.00 deg
[9/496] 11h_23m_41s\0008_49580.000Hz.pkl 🧈 | R: 0.0033 ± 0.0000 V | θ: 60.71 ± 0.00 deg
[10/496] 11h_23m_41s\0009_49590.000Hz.pkl 🧈 | R: 0.0033 ± 0.0000 V | θ: 60.69 ± 0.00 deg
[11/496] 11h_23m_41s\0010